# ProteinMPNN Inverse Folding

This notebook demonstrates both functions of the ProteinMPNN tool. In the first section, we use `run_proteinmpnn_sample` to generate new amino acid sequences conditioned on a target backbone structure. In the second section, we use `run_proteinmpnn_score` to evaluate how well an existing sequence fits a given structure by computing structure-conditioned likelihoods.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("proteinmpnn")
display_overview("proteinmpnn")
display_docs_section("proteinmpnn", "Background")

# ProteinMPNN

> ProteinMPNN is a deep learning model for protein sequence design given a protein backbone structure ("inverse folding"). It uses message passing neural networks to predict amino acid sequences that will fold into a target 3D structure. This module provides interfaces for *Sequence Sampling* (generating new sequences for a given backbone) and *Sequence Scoring* (evaluating how well a sequence fits a structure).

## Background

**What does this tool do?**
ProteinMPNN solves the "[inverse folding](https://en.wikipedia.org/wiki/Protein_design#Inverse_folding)" problem: given a protein backbone structure (the 3D coordinates of N, CA, C, O atoms), predict which amino acid sequence will fold into that structure. This is the inverse of structure prediction.

**Why is this important?**
Inverse folding enables:

* Protein engineering: Redesign natural proteins with improved stability, solubility, or expression.
* Scaffold design: Create proteins with desired shapes for binding, catalysis, or assembly.
* Therapeutic development: Design antibodies, enzymes, and binding proteins.
* Experimental validation: Generate multiple sequence candidates for a single structure to find the best experimental hits.

**Scientific foundation:**
ProteinMPNN uses a [message passing neural network](https://en.wikipedia.org/wiki/Graph_neural_network) architecture:

1. **Graph representation**: The protein backbone is represented as a graph where each residue is a node and edges connect spatially close residues (within \~30A).
2. **Message passing**: Information flows between connected residues over multiple rounds, allowing the model to learn long-range dependencies.
3. **[Autoregressive](https://en.wikipedia.org/wiki/Autoregressive_model) decoding**: Sequences are generated one residue at a time, conditioned on previously generated residues and the full structural context.
4. **Training**: The model was trained on \~19,000 protein structures from the [PDB](https://www.rcsb.org/) to maximize the probability of native sequences given their structures.

## Available tools

In [2]:
display_available_tools("proteinmpnn")

- **`run_proteinmpnn_sample()`** — Sample protein sequences using ProteinMPNN
- **`run_proteinmpnn_score()`** — Score protein sequences using ProteinMPNN

### `run_proteinmpnn_sample`

ProteinMPNN designs new protein sequences that are predicted to fold into a target backbone structure. We use Green Fluorescent Protein (GFP) as our target, a 238-residue beta-barrel protein whose well-characterized fold makes it an ideal test case for inverse folding. ProteinMPNN analyzes the 3D backbone coordinates and autoregressively generates amino acid sequences optimized for the target shape. The model also supports antibody-optimized weights via the `abmpnn` model choice.

In [3]:
from proto_tools import (
    InverseFoldingStructureInput,
    run_proteinmpnn_sample,
    ProteinMPNNSampleInput,
    ProteinMPNNSampleConfig,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_sample")

# Load the GFP structure and wrap it in an inverse folding input
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(structure=gfp_structure, chains_to_redesign=["A"])
inputs = ProteinMPNNSampleInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[InverseFoldingStructureInput]` | required | List of InverseFoldingInput objects, each containing a structure and optional chain\_ids/fixed\_positions constraints. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. Automatically converted to Structure. |
| `chain_ids` | `array` |  | Optional list of chain IDs to design. If None, all chains in the structure are designed. |
| `fixed_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions to keep fixed during design. Positions are 1-indexed. |

In [5]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_sample")

# Configure sampling | see docs above for all fields
config = ProteinMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs (closer to consensus)
    device="cuda",                  # Change to "cpu" if no GPU available
)

**Config** — `ProteinMPNNSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_choice` | `enum` | `proteinmpnn` | Model weights to use. `"proteinmpnn"` for the general-purpose ProteinMPNN model, `"abmpnn"` for antibody-optimized weights. Available options: `proteinmpnn`, `abmpnn` |
| `seed` | `integer` |  | Random seed to use for sampling. |
| `num_sequences_per_structure` | `integer` | `1` | Total number of sequences to generate per input structure. |
| `batch_size` | `integer` |  | Number of sequences to process simultaneously on GPU. Defaults to num\_sequences\_per\_structure. |
| `temperature` | `number` | `0.1` | Controls randomness in sampling from logits. |
| `excluded_amino_acids` | `array` |  | List of amino acids not allowed in the sequence. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution), or specific GPU devices like 'cuda:0'. Defaults to 'cuda'. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
result = run_proteinmpnn_sample(inputs, config)

Running run_proteinmpnn_sample [00:00]

In [7]:
# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_sample")

# Print the designed sequences
for i, seq_res in enumerate(result.designed_sequences):
    print(f"Structure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        preview = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"  Design {j}: {preview}")
        print(f"            Length: {len(seq)} residues")

**Output** — `InverseFoldingOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_sequences` | `List[DesignedSequences]` | required | List of DesignedSequences objects, one for each input structure. The order matches the input structures order. |
| `sequences` | `List[string]` | required | Designed amino acid sequences in single-letter code. |

Structure 0 designs:
  Design 1: SAGAALFTGVVPIKVELDGDVNGHKFSVEGEGEGDATTGTLNLHFVCTTG...
            Length: 227 residues
  Design 2: SKGAELFKGVVPILVDLDGDVNGHKFSVRGEGEGDATTGTLRLHFVCTTG...
            Length: 227 residues


### `run_proteinmpnn_score`

The scoring function evaluates how well a protein sequence fits a given backbone structure. It computes per-residue log-likelihoods using ProteinMPNN's structure-conditioned language model, producing metrics such as perplexity and average log-likelihood. Lower perplexity indicates better sequence-structure compatibility, making this useful for ranking candidate designs or evaluating natural sequences against their native structures.

Here we score one of the designed sequences from the sampling step above to evaluate how well it fits the GFP backbone.

In [8]:
from proto_tools import (
    ProteinMPNNScoringInput,
    ProteinMPNNScoringConfig,
    SequenceStructurePair,
    run_proteinmpnn_score,
)

In [9]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_score")

# Score the first designed sequence against the GFP backbone
designed_seq = result.designed_sequences[0].sequences[0]
pair = SequenceStructurePair(sequence=designed_seq, structure=gfp_structure)
scoring_input = ProteinMPNNScoringInput(sequence_structure_pairs=[pair])

**Input** — `ProteinMPNNScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence_structure_pairs` | `List[SequenceStructurePair]` | required | List of sequence-structure pairs to score. Each pair contains a sequence and a structure to score the sequence against. |
| `sequence` | `string` | required | Protein sequence to score against the structure. |
| `structure` | `Structure` | required | Protein structure to score the sequence against. |

In [10]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_score")

# Configure scoring | see docs above for all fields
scoring_config = ProteinMPNNScoringConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProteinMPNNScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `fixed_positions` | `Dict[string, List[integer]]` |  | Dictionary mapping chain IDs to fixed positions in the sequence. If None, no positions will be fixed. In scoring, fixed positions will not be utilized in perplexity calculation. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `model_choice` | `enum` | `proteinmpnn` | Model weights to use. `"proteinmpnn"` for the general-purpose model, `"abmpnn"` for antibody-optimized weights. Default: `"proteinmpnn"`. Available options: `proteinmpnn`, `abmpnn` |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"` (NVIDIA GPU), `"cpu"` (CPU execution). Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the scoring function
score_result = run_proteinmpnn_score(scoring_input, scoring_config)

Running run_proteinmpnn_score [00:00]

In [12]:
# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_score")

# Display scoring metrics for the designed sequence
score = score_result.scores[0]
print(f"Perplexity: {score['perplexity']:.3f}")
print(f"Log-likelihood: {score['log_likelihood']:.3f}")
print(f"Avg log-likelihood: {score['avg_log_likelihood']:.3f}")

**Output** — `InverseFoldingScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[InverseFoldingScoringMetrics]` | required | List of scoring outputs, one per input sequence-structure pair. Each entry is a `Metrics` subclass with scalar metrics (accessed via `score.perplexity` or `score["perplexity"]`) plus declared `logits` / `vocab` fields. |
| `logits` | `array` |  | Per-position logits array `(seq_len, vocab_size)`. `None` unless the tool returns logits. |
| `vocab` | `array` |  | Token ordering for `logits`. |
| `primary_metric` | `string` |  | Name of the metric that best summarizes the result overall (e.g. `"avg_plddt"` for AlphaFold2). Used by downstream UI and reporting to pick a headline value. |

Perplexity: 1.941
Log-likelihood: -150.489
Avg log-likelihood: -0.663


## Export Results

Designed sequences can be saved to standard file formats for downstream analysis, gene synthesis, or further computational evaluation such as structure prediction or molecular dynamics.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="proteinmpnn_designs", export_path=output_dir, file_format="fasta")
print(f"Sampling results exported to {output_dir / 'proteinmpnn_designs.fasta'}")

score_result.export(name="proteinmpnn_scores", export_path=output_dir, file_format="csv")
print(f"Scoring results exported to {output_dir / 'proteinmpnn_scores.csv'}")

Sampling results exported to example_output/proteinmpnn_designs.fasta
Scoring results exported to example_output/proteinmpnn_scores.csv
